In [18]:
from langchain.document_loaders.csv_loader import CSVLoader

In [22]:
loader = CSVLoader(file_path="prompts_db.csv")

data = loader.load()
len(data)

254

In [4]:
import tiktoken
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [5]:
num_tokens_from_string("tiktoken is great!", "cl100k_base")

6

In [6]:
data_list = [i.page_content for i in data]
data_string = ', '.join(str(i) for i in data_list)
len(data_string)

82849

In [8]:
for i in data:
    print(num_tokens_from_string(i.page_content, "cl100k_base"))

179
98
139
108
104
117
89
126
92
89
82
92
120
107
93
96
91
97
96
107
91
117
91
92
99
91
89
81
81
110
93
89
84
89
81
93
84
110
107
112
101
100
116
114
112
105
87
96
75
76
104
75
75
82
85
82
89
101
85
105
251
41
75
185
95
132
47
66
96
72
205
91
120
76
122
101
81
88
77
67
76
63
88
69
88
86
116
103
95
78
88
74
64
78
93
83
104
95
133
72
60
114
77
74
72
75
72
79
90
104
99
137
86
173
99
87
94
92
76
63
113
108
92
113
87
144
93
109
139
109
183
94
206
124
85
75
74
69
135
163
70
110
97
114
94
93
88
99
107
148
377
122
60
275
87
120
108
78
79
86
145
91
99
86
116
67


In [9]:
!pip install weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.3/120.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.3/215.3 kB 7.8 MB/s eta 0:00:00


In [ ]:
import weaviate
client = weaviate.Client(
  url="https://spark-e1l6ds2w.weaviate.network",
  auth_client_secret=weaviate.AuthApiKey("")
)

In [97]:
# clear this class first
client.schema.delete_class("Spark")
# lets make sure its vectorizer is what the one we want
# class_definition = {
#     "class": "Spark",
#     "vectorIndexConfig": {
#         "distance": "cosine" # Set to "cosine" for English models; "dot" for multilingual models
#     }
# }
# client.schema.create_class(class_definition)

In [99]:
!pip install -U langchain

In [79]:
import os
from langchain.embeddings import CohereEmbeddings
embeddings = CohereEmbeddings(cohere_api_key=os.environ['COHERE_API_KEY'], model="embed-english-light-v3.0")

In [7]:
import pandas as pd

df1 = pd.read_csv('prompts.csv')
df2 = pd.read_csv('prompts.csv', encoding='ISO-8859-1')

df = pd.concat([df1, df2]).drop_duplicates()

df.to_csv('prompts_db.csv')

In [9]:
url = "https://promptpile.com/prompts/641385b6963e420ee5244efa"

In [10]:
from langchain.document_loaders import WebBaseLoader
loader = WebBaseLoader(url)
data = loader.load()
data

[Document(page_content="\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nPrompt Pile - The AI Prompt Library\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nPrompt Pile\n\n\n\n\n\n\n\nSearch icon\n\n\nSearch\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\nOpen user menu\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n                                            Sign In\n\n\n\n\n\nOpen main menu\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\nFind a Designer\n\n\nAbout\n\n\n\n\n\n\n\n\nGomoku player\nAI-powered Gomoku player.\n\n\n            chatgpt\n        \n\n            strategy\n\n            board\n\n            game\n\n            ai\n\n\n\n\n\n\n\n\nCopy\n\n\nLet's play Gomoku. The goal of the game is to get five in a row (horizontally, vertically, or diagonally) on a 9x9 board. Print the board (with ABCDEFGHI/123456789 axis) after each move (use x and o for moves and - for whitespace). You and I take turns in moving, that is, make your move after my each move

In [11]:
from langchain.document_loaders import AsyncChromiumLoader
from langchain.document_transformers import BeautifulSoupTransformer
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo-0613")

In [12]:
from langchain.chains import create_extraction_chain

schema = {
    "properties": {
        "prompt_name": {"type": "string"},
        "prompt_content": {"type": "string"},
    },
    "required": ["prompt_name", "prompt_content"],
}


def extract(content: str, schema: dict):
    return create_extraction_chain(schema=schema, llm=llm).run(content)

In [14]:
import nest_asyncio
nest_asyncio.apply()

In [21]:
from langchain.embeddings import CohereEmbeddings
embeddings = CohereEmbeddings(model='embed-english-light-v3.0')

In [24]:
from langchain.vectorstores import Pinecone
import pinecone
import os

# initialize pinecone
pinecone.init(
    api_key=os.environ['PINECONE_API_KEY'],  # find at app.pinecone.io
    environment='us-west1-gcp',  # next to api key in console,
)

index_name = "spark-prompts"

docsearch = Pinecone.from_documents(data, embeddings, index_name=index_name)

In [ ]:
import pinecone
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

#Initialize embeddings & vectorstore
# embeddings = CohereEmbeddings(cohere_api_key=os.environ['COHERE_API_KEY'], model="embed-english-light-v3.0")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

pc = pinecone.Pinecone(
        api_key=''
    )

learn_index = pc.Index('sparklearn')
prompt_index = pc.Index('spark-prompts')

learnsearch = PineconeVectorStore(index=learn_index, embedding=embeddings)
promptsearch = PineconeVectorStore(index=prompt_index, embedding=embeddings)

learn_retriever = learnsearch.as_retriever(search_kwargs={"k": 8})
prompt_retriever = promptsearch.as_retriever(search_kwargs={"k": 8})

/opt/homebrew/Caskroom/miniforge/base/envs/chainlit/lib/python3.10/site-packages/pinecone/data/index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
learn_retriever.invoke('chain of thjoguht')

[Document(id='b67349c5-a7b7-45ea-841d-23083b4022e1', metadata={'language': 'en', 'source': 'http://promptingguide.ai//techniques/react', 'title': 'ReAct Prompting | Prompt Engineering Guide '}, page_content='t'),
 Document(id='52649079-0c89-45d5-9142-192c659d561e', metadata={'language': 'en', 'source': 'http://promptingguide.ai//models/chatgpt', 'title': 'ChatGPT Prompt Engineering | Prompt Engineering Guide '}, page_content='t'),
 Document(id='1c35873f-25a5-480e-8b98-44006eead59f', metadata={'language': 'en', 'source': 'http://promptingguide.ai//techniques/ape', 'title': 'Automatic Prompt Engineer (APE) | Prompt Engineering Guide '}, page_content='t'),
 Document(id='d917e635-30cb-460c-9cfe-ac8bbca5b306', metadata={'language': 'en', 'source': 'http://promptingguide.ai//applications/generating', 'title': 'Generating Data | Prompt Engineering Guide '}, page_content='t'),
 Document(id='f3e52d3e-0f69-4af9-8bae-fa11cd5514a9', metadata={'description': 'Learn how to use AI to organize data in